In [1]:
import pandas as pd
import json

print("Starting data harmonization...")

# 1. Load and Standardize Chest X-Ray
print("Processing Chest X-Ray data...")
with open("chest_xray_qa_1000.json", "r") as f:
    cxr_data = json.load(f)
cxr_df = pd.DataFrame(cxr_data)
# Rename columns to match standard
cxr_df = cxr_df[['image_id', 'question', 'answer']].rename(columns={'image_id': 'image_identifier'})
cxr_df['source'] = 'Chest_XRay'

# 2. Load and Standardize Slake
print("Processing Slake data...")
slake_df = pd.read_csv("slake_dataset.csv")
# Rename columns to match standard
slake_df = slake_df[['image_name', 'question', 'answer']].rename(columns={'image_name': 'image_identifier'})
slake_df['source'] = 'Slake'

# 3. Load and Standardize VQA-RAD
print("Processing VQA-RAD data...")
vqa_rad_df = pd.read_csv("vqa_rad.csv")
# VQA-RAD has 'image' column. We will rename it to standard
vqa_rad_df = vqa_rad_df[['image', 'question', 'answer']].rename(columns={'image': 'image_identifier'})
vqa_rad_df['source'] = 'VQA-RAD'

# 4. Merge them all together!
print("Merging all datasets...")
master_df = pd.concat([cxr_df, slake_df, vqa_rad_df], ignore_index=True)

# 5. Save the final Master Dataset
master_file = "master_qa_dataset.csv"
master_df.to_csv(master_file, index=False)

print(f"\n🎉 Success! Master dataset created with {len(master_df)} total Q&A pairs.")
print(f"Saved as: {master_file}")

# Display the first few rows so you can see your beautiful unified data!
display(master_df.head())

Starting data harmonization...
Processing Chest X-Ray data...
Processing Slake data...
Processing VQA-RAD data...
Merging all datasets...

🎉 Success! Master dataset created with 16821 total Q&A pairs.
Saved as: master_qa_dataset.csv


,image_identifier,question,answer,source
0,9a5094b2563a1ef3ff50dc5c7ff71345,Where is the Cardiomegaly?,The Cardiomegaly finding refers to an enlargem...,Chest_XRay
1,9a5094b2563a1ef3ff50dc5c7ff71345,Does this X-ray show Cardiomegaly?,"Yes, the Chest X-ray findings explicitly repor...",Chest_XRay
2,9a5094b2563a1ef3ff50dc5c7ff71345,How many findings are there?,There are three distinct findings listed: Card...,Chest_XRay
3,051132a778e61a86eb147c7c6f564dfe,Is there Cardiomegaly?,"Yes, the chest X-ray findings explicitly state...",Chest_XRay
4,051132a778e61a86eb147c7c6f564dfe,Does this X-ray show Cardiomegaly?,"Yes, the Chest X-ray findings explicitly list ...",Chest_XRay


In [2]:
import os
import pandas as pd
from datasets import load_dataset

# 1. Create a folder to save the actual VQA-RAD images
image_folder = "vqa_rad_images"
os.makedirs(image_folder, exist_ok=True)

print("Connecting to VQA-RAD to extract images...")
# 2. Load the dataset 
# (Replace "flaviagiammarino/vqa-rad" with your specific HF path if you used a different one!)
dataset = load_dataset("flaviagiammarino/vqa-rad", split="train")

fixed_data = []

print("Saving images and generating fixed paths...")
# 3. Loop through, save the image file, and record the correct path
for i, row in enumerate(dataset):
    file_name = f"vqa_rad_{i}.jpg"
    file_path = os.path.join(image_folder, file_name)
    
    # Save the actual image to your computer
    row["image"].save(file_path, format="JPEG")
    
    # Store the correct path and Q&A
    fixed_data.append({
        "image_identifier": file_path, # This is the REAL path now!
        "question": row["question"],
        "answer": row["answer"],
        "source": "VQA-RAD"
    })

# 4. Save the corrected data
vqa_rad_df = pd.DataFrame(fixed_data)
vqa_rad_df.to_csv("vqa_rad_fixed.csv", index=False)

print(f"\n🎉 Success! Saved {len(vqa_rad_df)} images to the '{image_folder}' folder.")
print("Fixed VQA-RAD dataset saved as 'vqa_rad_fixed.csv'")

# Display the fixed data to verify
display(vqa_rad_df.head())

c:\Users\santo\AppData\Local\Python\pythoncore-3.14-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Connecting to VQA-RAD to extract images...


Generating test split: 100%|██████████| 451/451 [00:00<00:00, 5688.48 examples/s]


Saving images and generating fixed paths...

🎉 Success! Saved 1793 images to the 'vqa_rad_images' folder.
Fixed VQA-RAD dataset saved as 'vqa_rad_fixed.csv'


,image_identifier,question,answer,source
0,vqa_rad_images\vqa_rad_0.jpg,are regions of the brain infarcted?,yes,VQA-RAD
1,vqa_rad_images\vqa_rad_1.jpg,are the lungs normal appearing?,no,VQA-RAD
2,vqa_rad_images\vqa_rad_2.jpg,which organ system is abnormal in this image?,cardiovascular,VQA-RAD
3,vqa_rad_images\vqa_rad_3.jpg,is the lesion causing significant brainstem he...,no,VQA-RAD
4,vqa_rad_images\vqa_rad_4.jpg,how was this image taken?,mri,VQA-RAD


In [ ]:
import pandas as pd
import json
import os

print("Creating Final Master Dataset with correct paths...")

# 1. Load the fixed VQA-RAD data
vqa_rad_df = pd.read_csv("vqa_rad_fixed.csv")

# 2. Load and format Slake data
slake_df = pd.read_csv("slake_dataset.csv")
slake_df = slake_df[['image_name', 'question', 'answer']].rename(columns={'image_name': 'image_identifier'})
# Note: Adjust "slake_images" if you named your Slake image folder something else!
slake_df['image_identifier'] = slake_df['image_identifier'].apply(lambda x: os.path.join("slake_images", x))
slake_df['source'] = 'Slake'

# 3. Load Chest X-Ray and Map IDs to ROCO images safely
with open("chest_xray_qa_1000.json", "r") as f:
    cxr_data = json.load(f)
cxr_df = pd.DataFrame(cxr_data)

# Extract unique question IDs and map them to our ROCO image folder
unique_ids = cxr_df['image_id'].unique()
id_to_path = {img_id: os.path.join("chest_xray_images", f"image_{i}.jpg") for i, img_id in enumerate(unique_ids)}

# Apply the mapping
cxr_df['image_identifier'] = cxr_df['image_id'].map(id_to_path)
cxr_df = cxr_df[['image_identifier', 'question', 'answer']]
cxr_df['source'] = 'Chest_XRay'

# 4. Merge them all together!
final_master_df = pd.concat([cxr_df, slake_df, vqa_rad_df], ignore_index=True)

# 5. Save the final robust Master Dataset
master_file = "final_master_qa_dataset.csv"
final_master_df.to_csv(master_file, index=False)

print(f"\n🎉 Success! Final Master Dataset created with {len(final_master_df)} Q&A pairs.")
print(f"Saved as: {master_file}")

# Let's peek at the final clean paths
display(final_master_df.sample(5))

Creating Final Master Dataset with correct paths...

🎉 Success! Final Master Dataset created with 16821 Q&A pairs.
Saved as: final_master_qa_dataset.csv


,image_identifier,question,answer,source
2119,slake_images\train_1119.jpg,What is the largest organ in the picture?,Lung,Slake
1012,slake_images\train_12.jpg,Does the picture contain lung?,Yes,Slake
8044,slake_images\train_7044.jpg,图中肿块位于肺的哪个部位?,左肺,Slake
12082,slake_images\validation_1247.jpg,图片中包含脾脏吗?,不包含,Slake
10220,slake_images\train_9220.jpg,图中有多少个器官?,5,Slake


: 

In [1]:
import pandas as pd
import json
import os
import uuid

print("Starting Data Harmonization...")
final_dataset = []

def generate_id(prefix):
    return f"{prefix}_{uuid.uuid4().hex[:8]}"

# ==========================================
# 1. ADDING TEXT-ONLY DATASETS
# ==========================================
print("Processing Text-only datasets...")

# Load ChatDoctor
try:
    df_chat = pd.read_json("chatdocter_Cleaned_data.json")
    for _, row in df_chat.iterrows():
        final_dataset.append({
            "id": generate_id("chatdoc"),
            "conversations": [
                {"from": "human", "value": str(row['input'])},
                {"from": "gpt", "value": str(row['output'])}
            ]
        })
except Exception as e:
    print(f"Skipping ChatDoctor: {e}")

# Load MedQuad
try:
    df_medquad = pd.read_json("Cleaned_medquad.json")
    for _, row in df_medquad.iterrows():
        final_dataset.append({
            "id": generate_id("medquad"),
            "conversations": [
                {"from": "human", "value": str(row['Question'])},
                {"from": "gpt", "value": str(row['Answer'])}
            ]
        })
except Exception as e:
    print(f"Skipping MedQuad: {e}")

# ==========================================
# 2. ADDING IMAGE-BASED DATASETS (VQA)
# ==========================================
print("Processing Image-based datasets...")

# Load Pneumonia X-Rays (from CSV)
try:
    df_pneumonia = pd.read_csv("xray_labels.csv")
    for _, row in df_pneumonia.iterrows():
        img_path = str(row['image_path'])
        label = str(row['label'])
        if os.path.exists(img_path):
            final_dataset.append({
                "id": generate_id("pneumonia"),
                "image": img_path,
                "conversations": [
                    # Notice the <image> tag! This tells the AI to look at the picture.
                    {"from": "human", "value": "<image>\nAnalyze this chest X-ray and provide a diagnosis."},
                    {"from": "gpt", "value": f"Based on the X-ray, the diagnosis is: {label}."}
                ]
            })
except Exception as e:
    print(f"Skipping Pneumonia data: {e}")

# Load Prescriptions (from previous AI_Training_Dataset if it exists)
try:
    with open("AI_Training_Dataset.json", "r", encoding="utf-8") as f:
        prescription_data = json.load(f)
        # Assuming these are already in the correct format from your previous notebook
        final_dataset.extend(prescription_data)
except Exception as e:
    print(f"Skipping Prescriptions: {e}")


# ==========================================
# 3. SAVE THE MASTER DATASET
# ==========================================
output_file = "Final_Multimodal_Dataset.json"

with open(output_file, "w", encoding="utf-8") as f:
    json.dump(final_dataset, f, indent=4)

print(f"\n🎉 Success! Master dataset created with {len(final_dataset)} total records.")
print(f"Saved to: {output_file}")

Starting Data Harmonization...
Processing Text-only datasets...
Processing Image-based datasets...

🎉 Success! Master dataset created with 279214 total records.
Saved to: Final_Multimodal_Dataset.json


In [ ]:
import torch

def check_device():
    # Check if CUDA is available
    if torch.cuda.is_available():
        print("✅ CUDA is available. GPU will be used.")
        device = torch.device("cuda")
    else:
        print("❌ CUDA is NOT available. CPU will be used.")
        device = torch.device("cpu")
    
    # Create a sample tensor on the selected device
    tensor = torch.zeros((2, 3), device=device)
    
    # Display device info
    print(f"Tensor is on device: {tensor.device}")
    
    # If GPU is available, show GPU name
    if device.type == "cuda":
        print(f"GPU Name: {torch.cuda.get_device_name(0)}")
        print(f"GPU Memory Allocated: {torch.cuda.memory_allocated(0)} bytes")
        print(f"GPU Memory Cached: {torch.cuda.memory_reserved(0)} bytes")

if __name__ == "__main__":
    check_device()
